# MedAssist — Fine-tune QLoRA bước chọn trích xuất triệu chứng (Kaggle GPU 16 GB)

- **Mục tiêu:** Fine-tune riêng bước **chọn** của trích xuất `phrases` (`extraction-phrases-v2` + `extraction-phrases-compact-v3`) theo `fine-tune-trich-xuat.md` bản 3.
- **Model gốc:** `aisingapore/Gemma-SEA-LION-v3-9B-IT` (QLoRA 4-bit với Unsloth).
- **Dữ liệu đầu vào:** upload `medassist-extraction-ft-data.zip` lên Kaggle Dataset (chứa `train.jsonl`, `val.jsonl`, `dev.jsonl`, `config.json`, `dataset_manifest.json`).
- **Đầu ra chính tại `/kaggle/working/`:** `medassist-extraction-q4_k_m.gguf`, `gguf_manifest.json`, `train_metrics.json`, `lora_adapter/`.

In [ ]:
import os
# Phải đặt trước khi import torch/unsloth để tránh lỗi DataParallel trên Kaggle GPU T4 x2
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install -q --upgrade unsloth

In [ ]:
import hashlib
import json
from pathlib import Path
import shutil
import torch

# Tự động tìm dataset trong /kaggle/input/ (kể cả khi Kaggle giải nén vào thư mục con)
def find_data_dir() -> Path:
    direct = [
        Path("/kaggle/input/medassist-extraction-ft"),
        Path("/kaggle/working/data"),
        Path("./data"),
    ]
    for d in direct:
        if (d / "train.jsonl").exists():
            return d
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        hits = sorted(kaggle_input.rglob("train.jsonl"))
        if hits:
            return hits[0].parent
    raise FileNotFoundError("Không tìm thấy train.jsonl trong /kaggle/input/ hoặc ./data/")

DATA_DIR = find_data_dir()
CFG_PATH = DATA_DIR / "config.json" if (DATA_DIR / "config.json").exists() else Path("config.json")
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")
TMP_WORK = Path("/tmp/medassist_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TMP_WORK.mkdir(parents=True, exist_ok=True)

cfg = json.loads(CFG_PATH.read_text(encoding="utf-8")) if CFG_PATH.exists() else {
    "seed": 20261122,
    "qlora": {
        "base_model": "aisingapore/Gemma-SEA-LION-v3-9B-IT",
        "load_in_4bit": True,
        "max_seq_length": 4096,
        "r": 16,
        "lora_alpha": 16,
        "lora_dropout": 0.0,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "learning_rate": 2e-4,
        "num_train_epochs": 2,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "warmup_ratio": 0.05,
        "lr_scheduler_type": "cosine",
        "optim": "adamw_8bit",
        "quantization_method": "q4_k_m",
    },
}
qcfg = cfg["qlora"]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

split_hashes = {}
print("DATA_DIR:", DATA_DIR, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
for name in ("train.jsonl", "val.jsonl", "dev.jsonl"):
    p = DATA_DIR / name
    split_hashes[name] = sha256_file(p)
    lines = sum(1 for l in p.open(encoding="utf-8") if l.strip())
    print(f"  {name}: lines={lines} | sha256={split_hashes[name]}")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=qcfg["base_model"],
    max_seq_length=qcfg["max_seq_length"],
    dtype=None,
    load_in_4bit=qcfg["load_in_4bit"],
)
model = FastLanguageModel.get_peft_model(
    model,
    r=qcfg["r"],
    target_modules=qcfg["target_modules"],
    lora_alpha=qcfg["lora_alpha"],
    lora_dropout=qcfg["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=cfg["seed"],
)

In [ ]:
from datasets import Dataset

def format_gemma_chat(example):
    """Khớp chính xác Modelfile Ollama của Gemma-SEA-LION-v3-9B-IT-GGUF:Q4_K_M."""
    msgs = example["messages"]
    user_text = msgs[0]["content"]
    assistant_text = msgs[1]["content"]
    return {
        "text": (
            f"<start_of_turn>user\n{user_text}<end_of_turn>\n"
            f"<start_of_turn>model\n{assistant_text}<end_of_turn>\n"
        )
    }

def load_jsonl_dataset(path: Path) -> Dataset:
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    ds = Dataset.from_list([{"id": r["id"], "messages": r["messages"]} for r in rows])
    return ds.map(format_gemma_chat, remove_columns=["messages"])

train_ds = load_jsonl_dataset(DATA_DIR / "train.jsonl")
val_ds = load_jsonl_dataset(DATA_DIR / "val.jsonl")
print("train:", len(train_ds), "| val:", len(val_ds))
print("sample prefix:", repr(train_ds[0]["text"][:120]))

In [ ]:
import inspect
from transformers import TrainingArguments, DataCollatorForSeq2Seq
import trl
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

# Lưu checkpoint tạm trong /tmp/medassist_work để không tràn quota 20 GB của /kaggle/working/
ckpt_dir = TMP_WORK / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)

sft_sig = inspect.signature(SFTTrainer.__init__).parameters
sft_kwargs = {
    "model": model,
    "train_dataset": train_ds,
    "eval_dataset": val_ds,
    "data_collator": DataCollatorForSeq2Seq(tokenizer=tokenizer),
    "args": TrainingArguments(
        output_dir=str(ckpt_dir),
        per_device_train_batch_size=qcfg["per_device_train_batch_size"],
        gradient_accumulation_steps=qcfg["gradient_accumulation_steps"],
        warmup_ratio=qcfg["warmup_ratio"],
        num_train_epochs=qcfg["num_train_epochs"],
        learning_rate=qcfg["learning_rate"],
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        optim=qcfg["optim"],
        lr_scheduler_type=qcfg["lr_scheduler_type"],
        seed=cfg["seed"],
        report_to="none",
    ),
}
sft_kwargs["processing_class" if "processing_class" in sft_sig else "tokenizer"] = tokenizer
if "dataset_text_field" in sft_sig:
    sft_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in sft_sig:
    sft_kwargs["max_seq_length"] = qcfg["max_seq_length"]
if "dataset_num_proc" in sft_sig:
    sft_kwargs["dataset_num_proc"] = 2
if "packing" in sft_sig:
    sft_kwargs["packing"] = False

trainer = SFTTrainer(**sft_kwargs)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

# Kiểm nhanh rằng chỉ phần assistant được tính loss
if "labels" in trainer.train_dataset.column_names:
    masked_len = sum(1 for x in trainer.train_dataset[0]["labels"] if x == -100)
    total_len = len(trainer.train_dataset[0]["labels"])
    print(f"Token masked (-100): {masked_len}/{total_len} | Response tokens: {total_len - masked_len} | warmup_steps={warmup_steps}/{total_steps}")

train_result = trainer.train()
eval_result = trainer.evaluate()
(WORK_DIR / "train_metrics.json").write_text(
    json.dumps({"train": train_result.metrics, "eval": eval_result}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Eval metrics:", eval_result)
shutil.rmtree(ckpt_dir, ignore_errors=True)

In [ ]:
# Kiểm nhanh độ hợp lệ JSON và khớp chính xác trên mẫu dev.jsonl trước khi xuất GGUF
FastLanguageModel.for_inference(model)
dev_rows = [json.loads(line) for line in (DATA_DIR / "dev.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()][:40]

json_ok = 0
exact_ok = 0
for r in dev_rows:
    user_prompt = r["messages"][0]["content"]
    gold = json.loads(r["messages"][1]["content"])
    prompt_str = f"<start_of_turn>user\n{user_prompt}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer([prompt_str], return_tensors="pt").to(model.device)
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=768, do_sample=False)
    ans = tokenizer.decode(gen[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).split("<end_of_turn>")[0].strip()
    try:
        pred = json.loads(ans)
        json_ok += 1
        if pred == gold:
            exact_ok += 1
    except Exception:
        pass

dev_check = {"dev_sample_n": len(dev_rows), "valid_json": json_ok, "exact_json": exact_ok}
print(f"Dev sample (n={len(dev_rows)}): valid_json={json_ok}/{len(dev_rows)} | exact_json={exact_ok}/{len(dev_rows)}")

In [ ]:
lora_dir = WORK_DIR / "lora_adapter"
gguf_tmp = TMP_WORK / "gguf_export"
gguf_tmp.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(lora_dir))
tokenizer.save_pretrained(str(lora_dir))

# Xuất GGUF tạm trong /tmp rồi chuyển file Q4_K_M duy nhất sang /kaggle/working/
model.save_pretrained_gguf(
    str(gguf_tmp),
    tokenizer,
    quantization_method=qcfg["quantization_method"],
)

candidates = sorted(gguf_tmp.rglob("*.gguf")) + sorted(Path(".").glob("*.gguf"))
q4_files = [p for p in candidates if "q4_k_m" in p.name.lower()] or candidates
assert q4_files, "Không tìm thấy file .gguf sau khi xuất"
src_gguf = q4_files[0]
dst_gguf = WORK_DIR / "medassist-extraction-q4_k_m.gguf"
shutil.move(str(src_gguf), str(dst_gguf))
shutil.rmtree(gguf_tmp, ignore_errors=True)

gguf_sha = sha256_file(dst_gguf)
manifest = {
    "gguf_file": dst_gguf.name,
    "size_bytes": dst_gguf.stat().st_size,
    "size_gib": round(dst_gguf.stat().st_size / (1024**3), 3),
    "gguf_sha256": gguf_sha,
    "base_model": qcfg["base_model"],
    "quantization_method": qcfg["quantization_method"],
    "data_sha256": split_hashes,
    "eval_loss": eval_result.get("eval_loss"),
    "dev_sample_check": dev_check,
}
(WORK_DIR / "gguf_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))